<a href="https://www.kaggle.com/code/lukaspanos/spaceship-titanic-log-reg-xg-boost?scriptVersionId=344989172" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

train = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')

pd.set_option('display.max_columns', None)
print(train.shape)
print(train.isnull().sum())


(8693, 14)
PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64


In [2]:
train.head(20)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
5,0005_01,Earth,False,F/0/P,PSO J318.5-22,44.0,False,0.0,483.0,0.0,291.0,0.0,Sandie Hinetthews,True
6,0006_01,Earth,False,F/2/S,TRAPPIST-1e,26.0,False,42.0,1539.0,3.0,0.0,0.0,Billex Jacostaffey,True
7,0006_02,Earth,True,G/0/S,TRAPPIST-1e,28.0,False,0.0,0.0,0.0,0.0,NaN,Candra Jacostaffey,True
8,0007_01,Earth,False,F/3/S,TRAPPIST-1e,35.0,False,0.0,785.0,17.0,216.0,0.0,Andona Beston,True
9,0008_01,Europa,True,B/1/P,55 Cancri e,14.0,False,0.0,0.0,0.0,0.0,0.0,Erraiam Flatic,True


In [3]:
train.drop(columns=['Name'], inplace=True)

In [4]:
train['Age'] = train['Age'].replace(0, np.nan)
train['Age'] = train['Age'].fillna(train['Age'].median())

Since there are not that many missing values in each column, I will remove all rows with NAN values. Then I will count the number of rows to see if we made a dent. 

In [5]:
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

# CryoSleep passengers can't spend → their NaN spending is 0
for col in spend_cols:
    train.loc[(train['CryoSleep'] == True) & (train[col].isnull()), col] = 0
    train[col] = train[col].fillna(train[col].median())

# Zero total spend + unknown CryoSleep → almost certainly asleep
train['TotalSpend'] = train[spend_cols].sum(axis=1)
train.loc[(train['CryoSleep'].isnull()) & (train['TotalSpend'] == 0), 'CryoSleep'] = True
train.loc[(train['CryoSleep'].isnull()) & (train['TotalSpend'] > 0), 'CryoSleep'] = False

# Remaining categoricals → mode; VIP → mode (it's 97% False anyway)
for col in ['HomePlanet', 'Destination', 'VIP']:
    train[col] = train[col].fillna(train[col].mode()[0])
train['Cabin'] = train['Cabin'].fillna('U/0/U')   # unknown deck/side become their own category

/tmp/ipykernel_16/1930273037.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train[col] = train[col].fillna(train[col].mode()[0])


In [6]:
for col in spend_cols + ['TotalSpend']:
    train[col] = np.log1p(train[col])

In [7]:
train.isnull().sum()

PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Transported     0
TotalSpend      0
dtype: int64

In [8]:
train[['Deck', 'CabinNum', 'Side']] = train['Cabin'].str.split('/', expand=True)
train['Group'] = train['PassengerId'].str.split('_').str[0]
train['GroupSize'] = train.groupby('Group')['Group'].transform('count')
train['IsAloneGroup'] = (train['GroupSize'] == 1).astype(int)
train.drop(columns = 'PassengerId', inplace=True)
print(train.groupby('GroupSize')['Transported'].mean())

GroupSize
1    0.452445
2    0.538050
3    0.593137
4    0.640777
5    0.592453
6    0.614943
7    0.541126
8    0.394231
Name: Transported, dtype: float64


In [9]:
train['NoSpend'] = (train['TotalSpend'] == 0).astype(int)
train['AwakeNoSpend'] = ((train['CryoSleep'] == 0) & (train['NoSpend'] == 1)).astype(int)

In [10]:
train['Destination'].unique()

array(['TRAPPIST-1e', 'PSO J318.5-22', '55 Cancri e'], dtype=object)

In [11]:
train['HomePlanet'].unique()

array(['Europa', 'Earth', 'Mars'], dtype=object)

In [12]:
train['Destination'] = train['Destination'].map({
    'TRAPPIST-1e': 1,
    'PSO J318.5-22': 2,
    '55 Cancri e': 3
})

In [13]:
train = pd.get_dummies(train, columns=['HomePlanet'], prefix='HP')

In [14]:
train.head(20)

,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,TotalSpend,Deck,CabinNum,Side,Group,GroupSize,IsAloneGroup,NoSpend,AwakeNoSpend,HP_Earth,HP_Europa,HP_Mars
0,False,B/0/P,1,39.0,False,0.000000,0.000000,0.000000,0.000000,0.000000,False,0.000000,B,0,P,0001,1,1,1,1,False,True,False
1,False,F/0/S,1,24.0,False,4.700480,2.302585,3.258097,6.309918,3.806662,True,6.602588,F,0,S,0002,1,1,0,0,True,False,False
2,False,A/0/S,1,58.0,True,3.784190,8.182280,0.000000,8.812248,3.912023,False,9.248021,A,0,S,0003,2,0,0,0,False,True,False
3,False,A/0/S,1,33.0,False,0.000000,7.157735,5.918894,8.110728,5.267858,False,8.551981,A,0,S,0003,2,0,0,0,False,True,False
4,False,F/1/S,1,16.0,False,5.717028,4.262680,5.023881,6.338594,1.098612,True,6.995766,F,1,S,0004,1,1,0,0,True,False,False
5,False,F/0/P,2,44.0,False,0.000000,6.182085,0.000000,5.676754,0.000000,True,6.652863,F,0,P,0005,1,1,0,0,True,False,False
6,False,F/2/S,1,26.0,False,3.761200,7.339538,1.386294,0.000000,0.000000,True,7.368340,F,2,S,0006,2,0,0,0,True,False,False
7,True,G/0/S,1,28.0,False,0.000000,0.000000,0.000000,0.000000,0.000000,True,0.000000,G,0,S,0006,2,0,1,0,True,False,False
8,False,F/3/S,1,35.0,False,0.000000,6.666957,2.890372,5.379897,0.000000,True,6.926577,F,3,S,0007,1,1,0,0,True,False,False
9,True,B/1/P,3,14.0,False,0.000000,0.000000,0.000000,0.000000,0.000000,True,0.000000,B,1,P,0008,3,0,1,0,False,True,False


In [15]:
train['Deck'].unique()

array(['B', 'F', 'A', 'G', 'U', 'E', 'D', 'C', 'T'], dtype=object)

In [16]:
train['Deck'] = train['Deck'].fillna(0)

In [17]:
train = pd.get_dummies(train, columns=['Deck'], prefix='Deck')


train['Side'] = train['Side'].map({
    'P': 1,
    'S': 2
})

train = train.drop(columns=['Cabin'])

train.head(20)

,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,TotalSpend,CabinNum,Side,Group,GroupSize,IsAloneGroup,NoSpend,AwakeNoSpend,HP_Earth,HP_Europa,HP_Mars,Deck_A,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Deck_U
0,False,1,39.0,False,0.000000,0.000000,0.000000,0.000000,0.000000,False,0.000000,0,1.0,0001,1,1,1,1,False,True,False,False,True,False,False,False,False,False,False,False
1,False,1,24.0,False,4.700480,2.302585,3.258097,6.309918,3.806662,True,6.602588,0,2.0,0002,1,1,0,0,True,False,False,False,False,False,False,False,True,False,False,False
2,False,1,58.0,True,3.784190,8.182280,0.000000,8.812248,3.912023,False,9.248021,0,2.0,0003,2,0,0,0,False,True,False,True,False,False,False,False,False,False,False,False
3,False,1,33.0,False,0.000000,7.157735,5.918894,8.110728,5.267858,False,8.551981,0,2.0,0003,2,0,0,0,False,True,False,True,False,False,False,False,False,False,False,False
4,False,1,16.0,False,5.717028,4.262680,5.023881,6.338594,1.098612,True,6.995766,1,2.0,0004,1,1,0,0,True,False,False,False,False,False,False,False,True,False,False,False
5,False,2,44.0,False,0.000000,6.182085,0.000000,5.676754,0.000000,True,6.652863,0,1.0,0005,1,1,0,0,True,False,False,False,False,False,False,False,True,False,False,False
6,False,1,26.0,False,3.761200,7.339538,1.386294,0.000000,0.000000,True,7.368340,2,2.0,0006,2,0,0,0,True,False,False,False,False,False,False,False,True,False,False,False
7,True,1,28.0,False,0.000000,0.000000,0.000000,0.000000,0.000000,True,0.000000,0,2.0,0006,2,0,1,0,True,False,False,False,False,False,False,False,False,True,False,False
8,False,1,35.0,False,0.000000,6.666957,2.890372,5.379897,0.000000,True,6.926577,3,2.0,0007,1,1,0,0,True,False,False,False,False,False,False,False,True,False,False,False
9,True,3,14.0,False,0.000000,0.000000,0.000000,0.000000,0.000000,True,0.000000,1,1.0,0008,3,0,1,0,False,True,False,False,True,False,False,False,False,False,False,False


In [18]:
train['Side'] = train['Side'].fillna(0)

In [19]:
train['CabinNum'] = train['CabinNum'].astype(float)

# Group is an ID with thousands of unique values — the signal was already
# extracted into GroupSize/IsAloneGroup. The raw ID is just noise. Drop it.
train.drop(columns=['Group'], inplace=True)

In [20]:
train['CryoSleep'] = train['CryoSleep'].astype(int)
train['VIP'] = train['VIP'].astype(int)
train['Transported'] = train['Transported'].astype(int)

train.head(20)

,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,TotalSpend,CabinNum,Side,GroupSize,IsAloneGroup,NoSpend,AwakeNoSpend,HP_Earth,HP_Europa,HP_Mars,Deck_A,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Deck_U
0,0,1,39.0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0,0.000000,0.0,1.0,1,1,1,1,False,True,False,False,True,False,False,False,False,False,False,False
1,0,1,24.0,0,4.700480,2.302585,3.258097,6.309918,3.806662,1,6.602588,0.0,2.0,1,1,0,0,True,False,False,False,False,False,False,False,True,False,False,False
2,0,1,58.0,1,3.784190,8.182280,0.000000,8.812248,3.912023,0,9.248021,0.0,2.0,2,0,0,0,False,True,False,True,False,False,False,False,False,False,False,False
3,0,1,33.0,0,0.000000,7.157735,5.918894,8.110728,5.267858,0,8.551981,0.0,2.0,2,0,0,0,False,True,False,True,False,False,False,False,False,False,False,False
4,0,1,16.0,0,5.717028,4.262680,5.023881,6.338594,1.098612,1,6.995766,1.0,2.0,1,1,0,0,True,False,False,False,False,False,False,False,True,False,False,False
5,0,2,44.0,0,0.000000,6.182085,0.000000,5.676754,0.000000,1,6.652863,0.0,1.0,1,1,0,0,True,False,False,False,False,False,False,False,True,False,False,False
6,0,1,26.0,0,3.761200,7.339538,1.386294,0.000000,0.000000,1,7.368340,2.0,2.0,2,0,0,0,True,False,False,False,False,False,False,False,True,False,False,False
7,1,1,28.0,0,0.000000,0.000000,0.000000,0.000000,0.000000,1,0.000000,0.0,2.0,2,0,1,0,True,False,False,False,False,False,False,False,False,True,False,False
8,0,1,35.0,0,0.000000,6.666957,2.890372,5.379897,0.000000,1,6.926577,3.0,2.0,1,1,0,0,True,False,False,False,False,False,False,False,True,False,False,False
9,1,3,14.0,0,0.000000,0.000000,0.000000,0.000000,0.000000,1,0.000000,1.0,1.0,3,0,1,0,False,True,False,False,True,False,False,False,False,False,False,False


In [21]:
numeric_cols = train.select_dtypes(include=['int64', 'float64']).columns
corr = train[numeric_cols].corr()['Transported'].sort_values(ascending=False)
print(corr)

Transported     1.000000
NoSpend         0.481628
CryoSleep       0.467230
Destination     0.108152
Side            0.093497
GroupSize       0.082644
AwakeNoSpend    0.056491
VIP            -0.037261
CabinNum       -0.043832
Age            -0.052951
IsAloneGroup   -0.113792
FoodCourt      -0.135029
ShoppingMall   -0.178536
VRDeck         -0.338688
RoomService    -0.356220
Spa            -0.361903
TotalSpend     -0.468941
Name: Transported, dtype: float64


In [22]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

X = train.drop(columns=['Transported'])
y = train['Transported']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)

y_pred = logreg.predict(X_test)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1:        {f1_score(y_test, y_pred):.3f}")

Accuracy:  0.777
Precision: 0.774
Recall:    0.788
F1:        0.781


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [23]:
print(X.isnull().sum()[X.isnull().sum() > 0])

Series([], dtype: int64)


In [24]:
print(X.columns.tolist())

['CryoSleep', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'TotalSpend', 'CabinNum', 'Side', 'GroupSize', 'IsAloneGroup', 'NoSpend', 'AwakeNoSpend', 'HP_Earth', 'HP_Europa', 'HP_Mars', 'Deck_A', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T', 'Deck_U']


In [25]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

xgb = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42)
print(f"XGB CV: {cross_val_score(xgb, X, y, cv=5, scoring='accuracy').mean():.4f}")

XGB CV: 0.7863


In [26]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [200, 400, 600],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'max_depth': [3, 4, 5, 6],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5]
}

rs = RandomizedSearchCV(XGBClassifier(random_state=42), param_dist,
                        n_iter=40, cv=5, scoring='accuracy', n_jobs=-1, random_state=42)
rs.fit(X, y)
print(rs.best_params_, round(rs.best_score_, 4))

{'subsample': 1.0, 'n_estimators': 200, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.7} 0.7957


In [27]:
test_ids = test['PassengerId']
test['Age'] = test['Age'].replace(0, np.nan)
test['Age'] = test['Age'].fillna(test['Age'].median())
for col in spend_cols:
    test.loc[(test['CryoSleep'] == True) & (test[col].isnull()), col] = 0
    test[col] = test[col].fillna(train[col].median())

test['TotalSpend'] = test[spend_cols].sum(axis=1)
test.loc[(test['CryoSleep'].isnull()) & (test['TotalSpend'] == 0), 'CryoSleep'] = True
test.loc[(test['CryoSleep'].isnull()) & (test['TotalSpend'] > 0), 'CryoSleep'] = False

for col in ['HomePlanet', 'Destination', 'VIP']:
    test[col] = test[col].fillna(test[col].mode()[0])
test['Cabin'] = test['Cabin'].fillna('U/0/U') 

for col in spend_cols + ['TotalSpend']:
    test[col] = np.log1p(test[col])

test[['Deck', 'CabinNum', 'Side']] = test['Cabin'].str.split('/', expand=True)
test['Group'] = test['PassengerId'].str.split('_').str[0]
test['GroupSize'] = test.groupby('Group')['Group'].transform('count')
test['IsAloneGroup'] = (test['GroupSize'] == 1).astype(int)
test.drop(columns = 'PassengerId', inplace=True)

test['NoSpend'] = (test['TotalSpend'] == 0).astype(int)
test['AwakeNoSpend'] = ((test['CryoSleep'] == 0) & (test['NoSpend'] == 1)).astype(int)

test['Destination'] = test['Destination'].map({
    'TRAPPIST-1e': 1,
    'PSO J318.5-22': 2,
    '55 Cancri e': 3
})

test = pd.get_dummies(test, columns=['HomePlanet'], prefix='HP')
test['Deck'] = test['Deck'].fillna(0)

test = pd.get_dummies(test, columns=['Deck'], prefix='Deck')


test['Side'] = test['Side'].map({
    'P': 1,
    'S': 2
})

test = test.drop(columns=['Cabin'])

test['Side'] = test['Side'].fillna(0)
test['CabinNum'] = test['CabinNum'].astype(float)
test.drop(columns=['Group'], inplace=True)

test['CryoSleep'] = test['CryoSleep'].astype(int)
test['VIP'] = test['VIP'].astype(int)



xgb_final = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)
xgb_final.fit(X, y)

test = test.reindex(columns=X.columns, fill_value=0)
print(test.isnull().sum().sum())   # must be 0 before predicting

preds = xgb_final.predict(test)

submission = pd.DataFrame({'PassengerId': test_ids, 'Transported': preds.astype(bool)})
submission.to_csv('submission.csv', index=False)
print(submission.head())

/tmp/ipykernel_16/46104084.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test[col] = test[col].fillna(test[col].mode()[0])


0
  PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01         True
